## Установка библиотек

In [1]:
# ⚠️ Запусти эту ячейку один раз в начале работы (или выполни команды в терминале)
# Рекомендуется создать виртуальное окружение: python3 -m venv venv && source venv/bin/activate

!pip3 install --upgrade pip
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu  # CPU-версия для Mac
!pip3 install ultralytics pandas tqdm numpy scipy scikit-learn pyyaml matplotlib
print("✅ Все зависимости установлены.")

Looking in indexes: https://download.pytorch.org/whl/cpu
✅ Все зависимости установлены.


## Импорты

In [2]:
import os
import sys
import time
import warnings
import pickle
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
import yaml

from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.utils.nms import non_max_suppression


from scipy.optimize import linear_sum_assignment
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")

# Настройки CPU (для M3)
device = "cpu"
torch.set_num_threads(4)

print(f"🚀 Устройство: {device}, потоков: {torch.get_num_threads()}")

# Пути
PROJECT_ROOT = Path("/Users/alexander/Developer/pcbcv")

DATASET_PATH = PROJECT_ROOT / "dataset"
DATA_YAML_PATH = DATASET_PATH / "data.yaml"

MODEL_PATH = (
    PROJECT_ROOT
    / "runs"
    / "detect"
    / "runs"
    / "m3_dropout_run"
    / "weights"
    / "best.pt"
)

RESULTS_DIR = PROJECT_ROOT / "mcd_results_v2"
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_YAML_PATH.exists(), f"Нет {DATA_YAML_PATH}"
assert MODEL_PATH.exists(), f"Нет {MODEL_PATH}"

with open(DATA_YAML_PATH) as f:
    class_names = yaml.safe_load(f)["names"]

print(f"Классы: {class_names}")

🚀 Устройство: cpu, потоков: 4
Классы: {0: 'mouse_bite', 1: 'spur', 2: 'missing_hole', 3: 'short', 4: 'open_circuit', 5: 'spurious_copper'}


## Запуск модели и проверка дропаута

In [3]:
# Загрузка модели
model = YOLO(str(MODEL_PATH))

# ВАЖНО:
# модель может возвращать:
# - tuple
# - dict(one2one / one2many)
# этот патч приводит всё к tensor

original_forward = model.model.forward

def fixed_forward(*args, **kwargs):
    out = original_forward(*args, **kwargs)

    # tuple -> predictions
    if isinstance(out, tuple):
        out = out[0]

    # dict -> one2one
    if isinstance(out, dict):

        if "one2one" in out:
            out = out["one2one"]

        elif "one2many" in out:
            out = out["one2many"]

    return out

model.model.forward = fixed_forward

# eval mode
model.model.eval()


# Поиск dropout слоёв
def count_dropout_layers(model_module):

    return sum(
        1
        for m in model_module.modules()
        if isinstance(m, torch.nn.modules.dropout._DropoutNd)
    )


dropout_count = count_dropout_layers(model.model)

if dropout_count == 0:
    raise RuntimeError(
        "❌ В модели нет Dropout слоёв! "
        "MCD бессмыслен."
    )

print(f"✅ Найдено {dropout_count} Dropout слоёв")


# Включаем ТОЛЬКО dropout
def enable_dropout(module):

    for m in module.modules():

        if isinstance(m, torch.nn.modules.dropout._DropoutNd):
            m.train()

✅ Найдено 1 Dropout слоёв


## Функции для оценки MCD

In [4]:
def iou(box1, box2):
    """box: [x1, y1, x2, y2]"""
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    if inter_x2 < inter_x1 or inter_y2 < inter_y1:
        return 0.0
    
    inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter_area
    
    return inter_area / union if union > 0 else 0.0

def cluster_boxes(all_boxes, iou_threshold=0.5):
    """
    Кластеризует боксы из разных прогонов.
    all_boxes: list of lists, каждый inner list = [x1, y1, x2, y2, conf, class_id]
    Возвращает: список кластеров (каждый кластер = список индексов)
    """
    if not all_boxes:
        return []
    
    # Собираем все боксы с меткой прогона
    flat_boxes = []
    for pass_idx, boxes in enumerate(all_boxes):
        for box in boxes:
            flat_boxes.append({
                'pass_idx': pass_idx,
                'box': box,
                'used': False
            })
    
    clusters = []
    for i in range(len(flat_boxes)):
        if flat_boxes[i]['used']:
            continue
        
        # Начинаем новый кластер
        cluster = [i]
        flat_boxes[i]['used'] = True
        
        for j in range(i + 1, len(flat_boxes)):
            if flat_boxes[j]['used']:
                continue
            
            box_i = flat_boxes[i]['box'][:4]
            box_j = flat_boxes[j]['box'][:4]
            
            if iou(box_i, box_j) >= iou_threshold:
                cluster.append(j)
                flat_boxes[j]['used'] = True
        
        clusters.append(cluster)
    
    return clusters

def compute_uncertainty_metrics(cluster_boxes):
    """
    Вычисляет метрики неопределённости для кластера.
    cluster_boxes: list of [conf, class_id]
    Возвращает: dict с метриками (без Mutual Information)
    """
    if not cluster_boxes:
        return {
            'entropy': 0.0,
            'variance_conf': 0.0,
            'mean_conf': 0.0,
            'consensus_class': -1,
            'num_detections': 0
        }
    
    confs = np.array([b[0] for b in cluster_boxes])
    classes = np.array([b[1] for b in cluster_boxes])
    num_classes = len(class_names)
    
    # 1. Энтропия распределения классов (правильная!)
    class_counts = np.bincount(classes, minlength=num_classes)
    class_probs = class_counts / len(classes)
    entropy = -np.sum(class_probs * np.log(class_probs + 1e-8))
    
    # 2. Дисперсия confidence (мера случайной неопределённости)
    variance_conf = np.var(confs) if len(confs) > 1 else 0.0
    
    # 3. Консенсусный класс (наиболее частый)
    consensus_class = np.argmax(class_counts)
    
    return {
        'entropy': entropy,
        'variance_conf': variance_conf,
        'mean_conf': np.mean(confs),
        'consensus_class': int(consensus_class),
        'num_detections': len(cluster_boxes)
    }

print("✅ Функции кластеризации и метрик загружены (MI удалён)")

✅ Функции кластеризации и метрик загружены (MI удалён)


## Код MCD

In [5]:
def mcd_predict_batched(
    model,
    image_paths,
    num_passes=30,
    raw_conf=0.15,
    final_conf=0.25,
    iou_cluster=0.5,
    batch_size=8,
):
    model.model.eval()
    enable_dropout(model.model)
    
    all_results = []
    n_batches = (len(image_paths) + batch_size - 1) // batch_size
    start_time = time.time()
    
    print(f"\n{'='*50}")
    print(f"MCD • {len(image_paths)} img • {num_passes} passes • {n_batches} batches")
    print(f"{'='*50}")
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        batch_paths = image_paths[start_idx:start_idx + batch_size]
        
        batch_results = {i: [] for i in range(len(batch_paths))}
        
        for pass_idx in range(num_passes):
            # preprocessing
            batch_imgs = []
            for path in batch_paths:
                img = cv2.imread(path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (640, 640))
                img = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0
                batch_imgs.append(img)
            
            batch = torch.stack(batch_imgs).to(device)
            
            with torch.no_grad():
                preds = model.model(batch)
            
            outputs = non_max_suppression(preds, conf_thres=raw_conf, iou_thres=1.0)
            
            for img_idx, res in enumerate(outputs):
                boxes = []
                if res is not None and len(res):
                    res = res.cpu().numpy()
                    for det in res:
                        boxes.append([float(x) for x in det[:6]])
                batch_results[img_idx].append(boxes)
            
            # live log каждые 5 passes
            if (pass_idx + 1) % 5 == 0:
                dets = sum(len(b) for br in batch_results.values() for b in br[-1:])
                print(f"  pass {pass_idx+1:2d}/{num_passes} • {dets:3d} raw dets", end='\r')
        
        # clustering
        for img_idx in range(len(batch_paths)):
            all_boxes = batch_results[img_idx]
            all_detections = [b for boxes in all_boxes for b in boxes if b[4] >= final_conf]
            
            # iou clustering
            used = [False] * len(all_detections)
            clusters = []
            for i in range(len(all_detections)):
                if used[i]:
                    continue
                cluster = [i]
                used[i] = True
                for j in range(i + 1, len(all_detections)):
                    if not used[j] and iou(all_detections[i][:4], all_detections[j][:4]) >= iou_cluster:
                        cluster.append(j)
                        used[j] = True
                clusters.append(cluster)
            
            cluster_metrics = []
            for cluster in clusters:
                cluster_data = [(all_detections[idx][4], int(all_detections[idx][5])) for idx in cluster]
                metrics = compute_uncertainty_metrics(cluster_data)
                avg_box = np.mean([all_detections[idx][:4] for idx in cluster], axis=0)
                cluster_metrics.append({
                    "bbox": avg_box.tolist(),
                    "class": metrics["consensus_class"],
                    "confidence": metrics["mean_conf"],
                    "entropy": metrics["entropy"],
                    "variance": metrics["variance_conf"],
                    "num_passes": len(cluster),
                })
            
            all_results.append(cluster_metrics)
        
        # batch summary
        elapsed = time.time() - start_time
        total_objs = sum(len(r) for r in all_results)
        print(f"\n✓ batch {batch_idx+1:3d}/{n_batches} • {total_objs:3d} objects • {elapsed:4.1f}s")
    
    model.model.eval()
    print(f"{'='*50}\n✅ DONE • {total_objs} objects • {elapsed:.1f}s\n")
    return all_results

## Подготовка данных

In [6]:
# ДОБАВЛЕНО: функция конвертации GT в абсолютные координаты
def convert_yolo_to_xyxy(gt_boxes, img_width=640, img_height=640):
    """
    Конвертирует GT из формата YOLO (нормализованный) в абсолютные xyxy.
    Для PCB обычно используется разрешение 640x640.
    """
    boxes_abs = []
    for gt in gt_boxes:
        # Нормализованные координаты
        x_center = gt['x_center']
        y_center = gt['y_center']
        width = gt['width']
        height = gt['height']
        
        # Конвертация в абсолютные
        abs_x1 = (x_center - width/2) * img_width
        abs_y1 = (y_center - height/2) * img_height
        abs_x2 = (x_center + width/2) * img_width
        abs_y2 = (y_center + height/2) * img_height
        
        boxes_abs.append({
            'bbox': [abs_x1, abs_y1, abs_x2, abs_y2],
            'class': gt['class'],
            'confidence': 1.0  # GT имеет уверенность 1.0
        })
    
    return boxes_abs

# Загрузка тестовых данных
test_img_dir = DATASET_PATH / "test" / "images"
test_lbl_dir = DATASET_PATH / "test" / "labels"
image_paths = sorted(list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.png")))
print(f"📸 Тестовых изображений: {len(image_paths)}")

# Парсим GT с конвертацией в абсолютные координаты
def parse_yolo_label_abs(label_path, img_width=640, img_height=640):
    """Парсит label файл и возвращает абсолютные координаты"""
    if not label_path.exists():
        return []
    
    boxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            
            class_id = int(parts[0])
            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])
            
            # Конвертация в абсолютные координаты
            x1 = (xc - w/2) * img_width
            y1 = (yc - h/2) * img_height
            x2 = (xc + w/2) * img_width
            y2 = (yc + h/2) * img_height
            
            boxes.append({
                'class': class_id,
                'bbox': [x1, y1, x2, y2]
            })
    
    return boxes

# Размер изображения (для PCB обычно 640x640, уточни у своего датасета)
IMAGE_SIZE = 640

ground_truth = []
for img_path in image_paths:
    label_path = test_lbl_dir / (img_path.stem + ".txt")
    gt_boxes = parse_yolo_label_abs(label_path, IMAGE_SIZE, IMAGE_SIZE)
    ground_truth.append(gt_boxes)

print(f"✅ Загружено {len(ground_truth)} изображений с GT в абсолютных координатах")
print(f"📊 Пример GT первого изображения: {ground_truth[0][:2] if ground_truth[0] else 'Нет дефектов'}")

📸 Тестовых изображений: 1068
✅ Загружено 1068 изображений с GT в абсолютных координатах
📊 Пример GT первого изображения: [{'class': 2, 'bbox': [228.25600000000003, 82.14399999999999, 259.168, 119.456]}, {'class': 2, 'bbox': [147.20000000000002, 252.832, 185.60000000000002, 290.144]}]


## Запуск MCD

In [7]:
!pip3 install tqdm tabulate

In [8]:
from tqdm import tqdm
from tabulate import tabulate
import time
NUM_PASSES = 30
RAW_CONF = 0.15
FINAL_CONF = 0.25
IOU_CLUSTER = 0.5
BATCH_SIZE = 4

print("🚀 Запуск MCD инференса...")
print(f"Конфиг: passes={NUM_PASSES}, raw_conf={RAW_CONF}, final_conf={FINAL_CONF}, iou={IOU_CLUSTER}")
print("-" * 60)

start_time = time.time()

mcd_predictions = mcd_predict_batched(
    model,
    [str(p) for p in image_paths],
    num_passes=NUM_PASSES,
    raw_conf=RAW_CONF,
    final_conf=FINAL_CONF,
    iou_cluster=IOU_CLUSTER,
    batch_size=BATCH_SIZE,
)

elapsed = time.time() - start_time

print("-" * 60)
print(f"✅ MCD инференс завершён за {elapsed:.2f} сек")

# ============================================
# классная ТАБЛИЦА МЕТРИК ПО ИЗОБРАЖЕНИЯМ
# ============================================
print("\n📊 ДЕТАЛЬНЫЕ МЕТРИКИ ПО ИЗОБРАЖЕНИЯМ:")
print("=" * 80)

summary_table = []
total_objects = 0
total_entropy = 0
total_variance = 0

for img_idx, clusters in enumerate(mcd_predictions):
    n_objects = len(clusters)
    total_objects += n_objects
    
    if n_objects > 0:
        mean_entropy = np.mean([c['entropy'] for c in clusters])
        mean_variance = np.mean([c['variance'] for c in clusters])
        mean_conf = np.mean([c['confidence'] for c in clusters])
    else:
        mean_entropy = mean_variance = mean_conf = 0
    
    total_entropy += mean_entropy
    total_variance += mean_variance
    
    summary_table.append([
        img_idx,
        n_objects,
        f"{mean_conf:.3f}",
        f"{mean_entropy:.4f}",
        f"{mean_variance:.4f}"
    ])

print(tabulate(
    summary_table,
    headers=["Изобр", "Объектов", "Ср.Conf", "Ср.Entropy", "Ср.Variance"],
    tablefmt="grid",
    numalign="right",
    stralign="center"
))

# ============================================
# АГРЕГИРОВАННЫЕ МЕТРИКИ
# ============================================
print("\n📈 АГРЕГИРОВАННАЯ СТАТИСТИКА:")
print("=" * 60)

n_images = len(mcd_predictions)
n_empty = sum(1 for c in mcd_predictions if len(c) == 0)

print(f"  Всего изображений:      {n_images}")
print(f"  Пустых (нет объектов):  {n_empty} ({n_empty/n_images*100:.1f}%)")
print(f"  Всего детекций (сумма): {total_objects}")
print(f"  Средн. объектов на изобр: {total_objects/n_images:.2f}")
print(f"  Средн. entropy:          {total_entropy/n_images:.4f}")
print(f"  Средн. variance conf:    {total_variance/n_images:.4f}")

# ============================================
# ПРИМЕР ПЕРВОГО ОБЪЕКТА (для проверки)
# ============================================
print("\n🔍 ПРИМЕР: первый объект на первом изображении:")
print("=" * 60)
if len(mcd_predictions) > 0 and len(mcd_predictions[0]) > 0:
    obj = mcd_predictions[0][0]
    print(f"  BBox:        [{obj['bbox'][0]:.1f}, {obj['bbox'][1]:.1f}, {obj['bbox'][2]:.1f}, {obj['bbox'][3]:.1f}]")
    print(f"  Class:       {obj['class']}")
    print(f"  Confidence:  {obj['confidence']:.3f}")
    print(f"  Entropy:     {obj['entropy']:.4f}")
    print(f"  Variance:    {obj['variance']:.4f}")
    print(f"  Num passes:  {obj['num_passes']}")
else:
    print("  Нет объектов для демонстрации")

print("\n" + "=" * 60)
print("✅ Готово")

🚀 Запуск MCD инференса...
Конфиг: passes=30, raw_conf=0.15, final_conf=0.25, iou=0.5
------------------------------------------------------------

MCD • 1068 img • 30 passes • 267 batches
  pass 30/30 •  69 raw dets
✓ batch   1/267 •   7 objects •  3.6s
  pass 30/30 •  48 raw dets
✓ batch   2/267 •  12 objects •  7.4s
  pass 30/30 •  45 raw dets
✓ batch   3/267 •  18 objects • 11.2s
  pass 30/30 •  62 raw dets
✓ batch   4/267 •  25 objects • 15.1s
  pass 30/30 •  40 raw dets
✓ batch   5/267 •  29 objects • 18.9s
  pass 30/30 •  57 raw dets
✓ batch   6/267 •  35 objects • 22.8s
  pass 30/30 •  62 raw dets
✓ batch   7/267 •  41 objects • 26.7s
  pass 30/30 •  59 raw dets
✓ batch   8/267 •  47 objects • 30.6s
  pass 30/30 •  69 raw dets
✓ batch   9/267 •  54 objects • 34.5s
  pass 30/30 •  69 raw dets
✓ batch  10/267 •  61 objects • 38.4s
  pass 30/30 •  54 raw dets
✓ batch  11/267 •  67 objects • 42.3s
  pass 30/30 •  59 raw dets
✓ batch  12/267 •  73 objects • 46.2s
  pass 30/30 •  50 r

## Сравнение с бейзлайном

In [34]:
from ultralytics.utils.ops import scale_boxes

# Обычный инференс (baseline) для сравнения - с простым resize как у MCD
baseline_preds = []
for img_path in tqdm(image_paths, desc="Baseline (simple resize)"):
    # Читаем и ресайзим КАК В MCD
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (640, 640))
    
    # Предсказание на ресайзнутом изображении
    results = model(img_resized, conf=FINAL_CONF, verbose=False)[0]
    
    boxes = []
    if results.boxes is not None:
        xyxy = results.boxes.xyxy.cpu().numpy()
        conf = results.boxes.conf.cpu().numpy()
        cls = results.boxes.cls.cpu().numpy().astype(int)
        
        for i in range(len(xyxy)):
            boxes.append({
                'bbox': xyxy[i].tolist(),  # координаты уже в правильном пространстве
                'class': int(cls[i]),
                'confidence': float(conf[i])
            })
    baseline_preds.append(boxes)

# Сохраняем всё в DataFrame
records = []
for idx, img_path in enumerate(image_paths):
    # MCD кластеры
    mcd_clusters = mcd_predictions[idx]
    mcd_bboxes = [c['bbox'] for c in mcd_clusters]
    mcd_confs = [c['confidence'] for c in mcd_clusters]
    mcd_classes = [c['class'] for c in mcd_clusters]
    mcd_entropy = np.mean([c['entropy'] for c in mcd_clusters]) if mcd_clusters else 0
    mcd_var = np.mean([c['variance'] for c in mcd_clusters]) if mcd_clusters else 0
    
    records.append({
        'image': img_path.name,
        'num_gt': len(ground_truth[idx]),
        'baseline_num_dets': len(baseline_preds[idx]),
        'mcd_num_dets': len(mcd_clusters),
        'mcd_mean_entropy': mcd_entropy,
        'mcd_mean_variance': mcd_var,
        'mcd_detections': mcd_clusters,   # для детального анализа
        'baseline_detections': baseline_preds[idx]
    })

df = pd.DataFrame(records)
df.to_csv(RESULTS_DIR / "mcd_full_results.csv", index=False)
print(f"Сохранено в {RESULTS_DIR / 'mcd_full_results.csv'}")

Baseline (simple resize): 100%|█████████████████████████████████████████████████████████| 1068/1068 [00:33<00:00, 31.70it/s]

Сохранено в /Users/alexander/Developer/pcbcv/mcd_results_v2/mcd_full_results.csv


## Метрики

In [20]:
!pip install supervision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [supervision]


In [36]:
from ultralytics.utils.metrics import ap_per_class
from ultralytics.utils.ops import scale_boxes
import supervision as sv
from supervision.metrics import MeanAveragePrecision
import numpy as np
import torch
import cv2

def compute_map_mcd(predictions, ground_truth, iou_threshold=0.5):
    import supervision as sv
    from supervision.metrics import MeanAveragePrecision
    import numpy as np
    
    detections_list = []
    targets_list = []
    
    for preds, gts in zip(predictions, ground_truth):
        # Конвертируем предсказания в supervision.Detections
        if preds and len(preds) > 0:
            boxes = np.array([p['bbox'] for p in preds], dtype=np.float32)
            scores = np.array([p['confidence'] for p in preds], dtype=np.float32)
            class_ids = np.array([p['class'] for p in preds], dtype=np.int64)
            detections_list.append(sv.Detections(
                xyxy=boxes,
                confidence=scores,
                class_id=class_ids
            ))
        else:
            detections_list.append(sv.Detections.empty())
        
        # Конвертируем GT
        if gts and len(gts) > 0:
            gt_boxes = np.array([g['bbox'] for g in gts], dtype=np.float32)
            gt_class_ids = np.array([g['class'] for g in gts], dtype=np.int64)
            targets_list.append(sv.Detections(
                xyxy=gt_boxes,
                class_id=gt_class_ids
            ))
        else:
            targets_list.append(sv.Detections.empty())
    
    metric = MeanAveragePrecision()
    metric.update(detections_list, targets_list)
    result = metric.compute()
    
    print(f"📊 Найдено {sum(len(d) for d in detections_list)} детекций и {sum(len(t) for t in targets_list)} GT боксов")
    
    return {
        'map_50': result.map50,
        'map_50_95': result.map50_95,
        'map_75': result.map75,
    }

# Конвертируем MCD предсказания в нужный формат
mcd_detections_formatted = []
for img_predictions in mcd_predictions:
    img_dets = []
    for cluster in img_predictions:
        img_dets.append({
            'bbox': cluster['bbox'],
            'class': cluster['class'],
            'confidence': cluster['confidence']
        })
    mcd_detections_formatted.append(img_dets)

# Baseline для сравнения - ИСПРАВЛЕННЫЙ (с простым resize как у MCD)
baseline_predictions = []
for img_path in tqdm(image_paths, desc="Baseline inference (simple resize)"):
    # Читаем и ресайзим КАК В MCD
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (640, 640))
    
    # Предсказание на ресайзнутом изображении
    results = model(img_resized, conf=FINAL_CONF, verbose=False)[0]
    
    boxes = []
    if results.boxes is not None:
        xyxy = results.boxes.xyxy.cpu().numpy()
        conf = results.boxes.conf.cpu().numpy()
        cls = results.boxes.cls.cpu().numpy().astype(int)
        
        for i in range(len(xyxy)):
            boxes.append({
                'bbox': xyxy[i].tolist(),
                'class': int(cls[i]),
                'confidence': float(conf[i])
            })
    baseline_predictions.append(boxes)

# Сравниваем метрики
print("\n📊 Сравнение качества детекции:")
print("="*50)

# MCD метрики
print("🔬 MCD (ансамбль с Dropout):")
mcd_metrics = compute_map_mcd(mcd_detections_formatted, ground_truth)
if mcd_metrics:
    print(f"   • mAP@0.5: {mcd_metrics['map_50']:.4f}")
    print(f"   • mAP@0.5:0.95: {mcd_metrics['map_50_95']:.4f}")
    print(f"   • mAP@0.75: {mcd_metrics['map_75']:.4f}")

# Baseline метрики
print("\n📌 Baseline (стандартный инференс с простым resize):")
baseline_metrics = compute_map_mcd(baseline_predictions, ground_truth)
if baseline_metrics:
    print(f"   • mAP@0.5: {baseline_metrics['map_50']:.4f}")
    print(f"   • mAP@0.5:0.95: {baseline_metrics['map_50_95']:.4f}")
    print(f"   • mAP@0.75: {baseline_metrics['map_75']:.4f}")

Baseline inference (simple resize): 100%|███████████████████████████████████████████████| 1068/1068 [00:33<00:00, 31.86it/s]



📊 Сравнение качества детекции:
🔬 MCD (ансамбль с Dropout):
📊 Найдено 1692 детекций и 1662 GT боксов
   • mAP@0.5: 0.9776
   • mAP@0.5:0.95: 0.5525
   • mAP@0.75: 0.5606

📌 Baseline (стандартный инференс с простым resize):
📊 Найдено 2014 детекций и 1662 GT боксов
   • mAP@0.5: 0.9541
   • mAP@0.5:0.95: 0.5211
   • mAP@0.75: 0.5038


## Сводка

In [37]:
# Сохраняем все результаты
import json

# Подготовка DataFrame для анализа
records = []
for idx, (img_path, mcd_clusters, gt) in enumerate(zip(image_paths, mcd_predictions, ground_truth)):
    records.append({
        'image': img_path.name,
        'num_gt': len(gt),
        'mcd_num_dets': len(mcd_clusters),
        'mcd_mean_entropy': np.mean([c['entropy'] for c in mcd_clusters]) if mcd_clusters else 0,
        'mcd_mean_variance': np.mean([c['variance'] for c in mcd_clusters]) if mcd_clusters else 0,
        'mcd_mean_conf': np.mean([c['confidence'] for c in mcd_clusters]) if mcd_clusters else 0,
        'baseline_num_dets': len(baseline_predictions[idx])
    })

df_results = pd.DataFrame(records)
df_results.to_csv(RESULTS_DIR / "mcd_comparison_results.csv", index=False)

# Сохраняем детальные предсказания в JSON
with open(RESULTS_DIR / "mcd_predictions.json", 'w') as f:
    json.dump(mcd_detections_formatted, f, indent=2)

# Финальный отчёт
report = f"""
{'='*60}
ФИНАЛЬНЫЙ ОТЧЁТ: MC DROPOUT ДЛЯ YOLOv8n
{'='*60}

ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:
• Прогонов MC Dropout: {NUM_PASSES}
• Raw порог уверенности: {RAW_CONF}
• Финальный порог: {FINAL_CONF}
• Порог IoU кластеризации: {IOU_CLUSTER}
• Размер батча: {BATCH_SIZE}

СТАТИСТИКА ДЕТЕКЦИИ:
• Всего изображений: {len(df_results)}
• Изображений с дефектами: {df_results['num_gt'].gt(0).sum()}
• Среднее GT детекций: {df_results['num_gt'].mean():.2f}

• Baseline (одиночный прогон):
    - Среднее детекций: {df_results['baseline_num_dets'].mean():.2f}
    - Медиана детекций: {df_results['baseline_num_dets'].median():.0f}

• MCD (ансамбль):
    - Среднее детекций: {df_results['mcd_num_dets'].mean():.2f}
    - Медиана детекций: {df_results['mcd_num_dets'].median():.0f}

НЕОПРЕДЕЛЁННОСТЬ MCD:
• Средняя энтропия: {df_results['mcd_mean_entropy'].mean():.4f}
• Средняя дисперсия confidence: {df_results['mcd_mean_variance'].mean():.4f}
• Средняя уверенность: {df_results['mcd_mean_conf'].mean():.4f}

МЕТРИКИ КАЧЕСТВА:
"""

if mcd_metrics and baseline_metrics:
    report += f"""
• mAP@0.5:
    - Baseline: {baseline_metrics['map_50']:.4f}
    - MCD: {mcd_metrics['map_50']:.4f}
    - Изменение: {((mcd_metrics['map_50'] - baseline_metrics['map_50']) / max(baseline_metrics['map_50'], 1e-6) * 100):+.1f}%

• mAP@0.5:0.95:
    - Baseline: {baseline_metrics['map_50_95']:.4f}
    - MCD: {mcd_metrics['map_50_95']:.4f}
    - Изменение: {((mcd_metrics['map_50_95'] - baseline_metrics['map_50_95']) / max(baseline_metrics['map_50_95'], 1e-6) * 100):+.1f}%

• mAP@0.75:
    - Baseline: {baseline_metrics['map_75']:.4f}
    - MCD: {mcd_metrics['map_75']:.4f}
    - Изменение: {((mcd_metrics['map_75'] - baseline_metrics['map_75']) / max(baseline_metrics['map_75'], 1e-6) * 100):+.1f}%
"""

report += f"\n{'='*60}\n"
report += f"📁 Результаты сохранены в: {RESULTS_DIR}\n"
report += f"   - mcd_comparison_results.csv\n"
report += f"   - mcd_predictions.json\n"
report += f"   - summary_report.txt\n"

# Сохраняем отчёт
with open(RESULTS_DIR / "summary_report.txt", 'w') as f:
    f.write(report)

print(report)
print("\n🎉 Анализ завершён! Результаты сохранены.")


ФИНАЛЬНЫЙ ОТЧЁТ: MC DROPOUT ДЛЯ YOLOv8n

ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:
• Прогонов MC Dropout: 30
• Raw порог уверенности: 0.15
• Финальный порог: 0.25
• Порог IoU кластеризации: 0.5
• Размер батча: 4

СТАТИСТИКА ДЕТЕКЦИИ:
• Всего изображений: 1068
• Изображений с дефектами: 829
• Среднее GT детекций: 1.56

• Baseline (одиночный прогон):
    - Среднее детекций: 1.89
    - Медиана детекций: 2

• MCD (ансамбль):
    - Среднее детекций: 1.58
    - Медиана детекций: 2

НЕОПРЕДЕЛЁННОСТЬ MCD:
• Средняя энтропия: 0.0002
• Средняя дисперсия confidence: 0.0085
• Средняя уверенность: 0.4815

МЕТРИКИ КАЧЕСТВА:

• mAP@0.5:
    - Baseline: 0.9541
    - MCD: 0.9776
    - Изменение: +2.5%

• mAP@0.5:0.95:
    - Baseline: 0.5211
    - MCD: 0.5525
    - Изменение: +6.0%

• mAP@0.75:
    - Baseline: 0.5038
    - MCD: 0.5606
    - Изменение: +11.3%

📁 Результаты сохранены в: /Users/alexander/Developer/pcbcv/mcd_results_v2
   - mcd_comparison_results.csv
   - mcd_predictions.json
   - summary_report.txt


🎉 Ана